# EduPulse — 개별 모델 하이퍼파라미터 실험

각 모델의 하이퍼파라미터를 변경하며 성능 변화를 실험합니다.

**실험 대상:**
1. XGBoost: n_estimators, max_depth, learning_rate
2. Prophet: seasonality_mode, changepoint_prior_scale
3. LSTM: hidden_size, num_layers, epochs, learning_rate
4. 최적 하이퍼파라미터 정리

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

import os
data_path = '../edupulse/data/warehouse/training_dataset.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    from edupulse.preprocessing.merger import build_training_dataset
    df = build_training_dataset()

print(f'데이터: {df.shape}')

---
## 1. XGBoost 하이퍼파라미터 실험

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from edupulse.model.xgboost_model import FEATURE_COLUMNS, TARGET_COLUMN

available = [c for c in FEATURE_COLUMNS if c in df.columns]
X = df[available].fillna(0).values
y = df[TARGET_COLUMN].values

def xgb_mape(params, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    mapes = []
    for train_idx, val_idx in tscv.split(X):
        model = XGBRegressor(random_state=42, n_jobs=-1, **params)
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        nonzero = y[val_idx] != 0
        if nonzero.any():
            mape = np.mean(np.abs((y[val_idx][nonzero] - preds[nonzero]) / y[val_idx][nonzero])) * 100
            mapes.append(mape)
    return np.mean(mapes) if mapes else float('nan')

In [ ]:
# n_estimators 실험
n_est_values = [50, 100, 200, 300, 500]
n_est_results = []
for n in n_est_values:
    mape = xgb_mape({'n_estimators': n, 'max_depth': 4, 'learning_rate': 0.05})
    n_est_results.append(mape)
    print(f'n_estimators={n:4d}: MAPE {mape:.2f}%')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_est_values, n_est_results, 'o-', color='steelblue')
ax.set_xlabel('n_estimators')
ax.set_ylabel('MAPE (%)')
ax.set_title('XGBoost: n_estimators vs MAPE')
plt.tight_layout()
plt.show()

In [ ]:
# max_depth 실험
depth_values = [2, 3, 4, 5, 6, 8]
depth_results = []
for d in depth_values:
    mape = xgb_mape({'n_estimators': 300, 'max_depth': d, 'learning_rate': 0.03})
    depth_results.append(mape)
    print(f'max_depth={d}: MAPE {mape:.2f}%')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depth_values, depth_results, 'o-', color='coral')
ax.set_xlabel('max_depth')
ax.set_ylabel('MAPE (%)')
ax.set_title('XGBoost: max_depth vs MAPE')
plt.tight_layout()
plt.show()

In [ ]:
# learning_rate 실험
lr_values = [0.01, 0.03, 0.05, 0.1, 0.2]
lr_results = []
for lr in lr_values:
    mape = xgb_mape({'n_estimators': 300, 'max_depth': 4, 'learning_rate': lr})
    lr_results.append(mape)
    print(f'learning_rate={lr:.2f}: MAPE {mape:.2f}%')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lr_values, lr_results, 'o-', color='green')
ax.set_xlabel('learning_rate')
ax.set_ylabel('MAPE (%)')
ax.set_title('XGBoost: learning_rate vs MAPE')
plt.tight_layout()
plt.show()

---
## 2. Prophet 하이퍼파라미터 실험

In [ ]:
try:
    from edupulse.model.prophet_model import ProphetForecaster
    PROPHET_OK = True
except ImportError:
    PROPHET_OK = False
    print('Prophet 미설치 — 이 섹션 건너뜀')

In [ ]:
if PROPHET_OK:
    # seasonality_mode 실험
    for mode in ['additive', 'multiplicative']:
        model = ProphetForecaster(seasonality_mode=mode)
        result = model.evaluate(df, n_splits=3)
        print(f'seasonality_mode={mode:16s}: MAPE {result["mape"]:.2f}%')

In [ ]:
if PROPHET_OK:
    # changepoint_prior_scale 실험
    cp_values = [0.01, 0.05, 0.1, 0.15, 0.3, 0.5]
    cp_results = []
    for cp in cp_values:
        model = ProphetForecaster(changepoint_prior_scale=cp)
        result = model.evaluate(df, n_splits=3)
        cp_results.append(result['mape'])
        print(f'changepoint_prior_scale={cp:.2f}: MAPE {result["mape"]:.2f}%')
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(cp_values, cp_results, 'o-', color='purple')
    ax.set_xlabel('changepoint_prior_scale')
    ax.set_ylabel('MAPE (%)')
    ax.set_title('Prophet: changepoint_prior_scale vs MAPE')
    plt.tight_layout()
    plt.show()

---
## 3. LSTM 하이퍼파라미터 실험

LSTM은 학습 시간이 길어서 간소화된 실험을 진행합니다.

In [ ]:
try:
    from edupulse.model.lstm_model import LSTMForecaster
    from edupulse.model.utils import get_device
    LSTM_OK = True
    print(f'디바이스: {get_device()}')
except ImportError:
    LSTM_OK = False
    print('PyTorch 미설치 — 이 섹션 건너뜀')

In [ ]:
if LSTM_OK:
    # epochs 실험 (학습 곡선 관찰)
    epoch_values = [20, 50, 100]
    for ep in epoch_values:
        model = LSTMForecaster()
        model.train(df, epochs=ep, augment=True)
        # 빠른 평가 (2-fold)
        result = model.evaluate(df, n_splits=2)
        print(f'epochs={ep:4d}: MAPE {result["mape"]:.2f}%')

In [ ]:
if LSTM_OK:
    # augment 효과 실험
    for aug in [False, True]:
        model = LSTMForecaster()
        model.train(df, epochs=50, augment=aug)
        result = model.evaluate(df, n_splits=2)
        label = 'augment=True ' if aug else 'augment=False'
        print(f'{label}: MAPE {result["mape"]:.2f}%')

---
## 4. 최적 하이퍼파라미터 정리

In [ ]:
print('=' * 50)
print('  최적 하이퍼파라미터 정리')
print('=' * 50)
print()
print('XGBoost:')
print('  위 실험 결과에서 가장 낮은 MAPE 조합을 선택하세요.')
print('  기본값: n_estimators=300, max_depth=4, learning_rate=0.03')
print()
if PROPHET_OK:
    print('Prophet:')
    print('  기본값: seasonality_mode=multiplicative, changepoint_prior_scale=0.15')
    print()
if LSTM_OK:
    print('LSTM:')
    print('  기본값: epochs=100, hidden=64, layers=2, dropout=0.3')
    print('  증강(augment=True) 권장')
print()
print('최적값을 각 모델 파일의 기본 파라미터에 반영하세요.')